# SmolVLA fine-tuning (Colab, resumable) — ADR-0029 / Week 2

**Constraints (owner/ADR-0029):** GPU = Google Colab (T4-class, sessions may disconnect). Training MUST be resumable from checkpoints stored on **Google Drive**. **Inference runs locally** from exported checkpoint; **no Colab tunnel dependency**. Do not tune on validation/test to rescue gate.

> **Status:** template/prep. Verify LeRobot/SmolVLA config (dims, camera keys, normalization, action space) against `docs/VLA_NOTES.md` and live `policy.config` before a real run. Record exact package versions and runtime.

## 0) Mount Drive + set paths

In [ ]:
# Google auth/mount is a HUMAN STEP if interactive approval is required.
from google.colab import drive  # type: ignore

drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/vlarobot_smolvla")
DATASET_DIR = DRIVE_ROOT / "dataset"  # upload/copy collected demos here
CKPT_DIR = DRIVE_ROOT / "checkpoints"  # resumable checkpoints (persisted)
EXPORT_DIR = DRIVE_ROOT / "export"  # local-inference export
for p in [DATASET_DIR, CKPT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT=", DRIVE_ROOT)

## 1) Install (pin versions; record output)
Use a compatible `torch`/`lerobot`/`transformers` set. Prefer the versions validated in `docs/VLA_NOTES.md` / `docs/VERSIONS.md`.

In [ ]:
# Example (VERIFY before use):
# !pip install -q "lerobot[smolvla]" transformers accelerate
# !pip freeze | grep -E 'lerobot|torch|transformers|accelerate'
print("TODO: install pinned stack and record versions")

## 2) Inspect base config (MEASURED, must be recorded)

In [ ]:
# from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
# model_id = 'lerobot/smolvla_base'
# policy = SmolVLAPolicy.from_pretrained(model_id)
# print(policy.config.input_features)
# print(policy.config.output_features)
# TODO: save config summary to DRIVE_ROOT/'policy_config.txt' and update docs/VLA_NOTES.md

## 3) Dataset
Point to the collected demo dataset (LeRobot-compatible as produced/converted by `collect-demos`). Confirm camera keys, state dim, action dim (7-D EE-delta + gripper mapped per VLA_NOTES), and task/instruction field.

In [ ]:
DATASET_REPO_OR_PATH = str(DATASET_DIR)  # adjust to actual dataset id/path
print("dataset=", DATASET_REPO_OR_PATH)

## 4) Train (resumable from Drive)
Use LeRobot's training entrypoint/config for SmolVLA fine-tuning. **Always** set output/checkpoint dir inside `CKPT_DIR` on Drive and enable resume-from-latest when rerun after disconnect.

In [ ]:
# PSEUDOCOMMAND — adapt to the installed LeRobot CLI/API and verified config:
# !python -m lerobot.scripts.train \
#   --policy.path=lerobot/smolvla_base \
#   --dataset.repo_id=<DATASET_REPO_OR_PATH> \
#   --output_dir={CKPT_DIR} \
#   --resume=true \
#   --steps=<N> --save_freq=<N> --log_freq=<N>
# Record: exact command, steps/epochs, batch size, runtime, and final metrics.
print("TODO: run resumable training with checkpoints on Drive")

## 5) Export for LOCAL inference (no Colab tunnel)
Copy/export the selected checkpoint to `EXPORT_DIR` (Drive), then download/sync to the local machine for `ModelServer` inference.

In [ ]:
# TODO: export/save_pretrained or copy best checkpoint to EXPORT_DIR
# Verify locally: load checkpoint, run preprocess->select_action->postprocess on a sample obs.
print("export_dir=", EXPORT_DIR)

## HUMAN STEP (only if unavoidable)
If Google Drive mount requires interactive auth in this session: (1) run cell 0, complete browser auth once, confirm `DRIVE_ROOT` exists; (2) ensure dataset is present at `DATASET_DIR`; (3) re-run from cell 4 — it resumes from latest checkpoint in `CKPT_DIR`. No tunnel, no ongoing interaction required after auth.